In [ ]:
from frumkin.electrolyte import LatticeElectrolyte, Ion, Water
from frumkin.gongadze_iglic import GongadzeIglic
import numpy as np 
from scipy import constants as C 

PZC_SHE = 0.0
RHE_SHIFT = 59e-3

def get_model(conc, cation_size=6, ohp=4, min_eps=5):
    el = LatticeElectrolyte([
        Water(min_eps=min_eps),
        Ion(name=r"Na$^+$", size=cation_size, concentration=conc, charge=+1),
        Ion(name=r"Cl$^-$", size=2, concentration=conc, charge=-1),
    ])
    model = GongadzeIglic(el, ohp=ohp)
    return model

def fbv(potential_she, conc, cation_size=6, ohp=4, min_eps=5, xprime=3, coverage=None):
    phi0 = potential_she - PZC_SHE 

    model = get_model(conc, cation_size, ohp, min_eps)
    result = model.voltammetry(phi0)

    phi2 = phi0 - result.electric_field * xprime 
    beta_e0 = C.elementary_charge / (C.Boltzmann * 298)  # e0/kbT
    alpha = 0.5
    if coverage is None:  # Volmer rds
        return -np.exp(-beta_e0 * alpha * (phi0 - phi2))
    if coverage is not None:  # Heyrovsky rds
        return -coverage * np.exp(-beta_e0 * alpha * (phi0 - phi2))

concentrations = [10e-3, 50e-3, 100e-3, 1000e-3]
ph_values = [11, 12, 13]
potential_rhe = np.linspace(-0.2, np.max(ph_values) * RHE_SHIFT + PZC_SHE, 100)

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib widget 

ACIDITY = 11 

fig = plt.figure(figsize=(4, 3))
ax = fig.add_subplot()

currents = [] 

for c in concentrations: 
    potential_she = potential_rhe - RHE_SHIFT * ACIDITY 
    i = fbv(potential_she, c)
    currents.append(i) 

for c, i in zip(concentrations, currents):
    ax.plot(potential_rhe, i / np.max(np.abs(currents)), label=f"{c*1e3:.0f}mM")

ax.legend()
ax.set_xlim([-0.25, 0.15])
ax.set_ylabel(r"$j/j_\mathrm{max}$")
ax.set_xlabel(r"$E$ vs. RHE / V")

fig.tight_layout()

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib widget 

CONCENTRATION = 100e-3 

COVERAGE = {
    11: 0.8,
    12: 0.4,
    13: 0.2,
}

fig = plt.figure(figsize=(4, 3))
ax = fig.add_subplot()

currents = [] 

for ph in ph_values: 
    potential_she = potential_rhe - RHE_SHIFT * ph 
    i = fbv(potential_she, CONCENTRATION, coverage=COVERAGE[ph])
    currents.append(i) 

for ph, i in zip(ph_values, currents):
    ax.plot(potential_rhe, i / np.max(np.abs(currents)), label=f"pH {ph:.0f}")

ax.legend()
ax.set_xlim([-0.25, 0.15])
ax.set_ylabel(r"$j/j_\mathrm{max}$")
ax.set_xlabel(r"$E$ vs. RHE / V")

fig.tight_layout()